In [104]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [105]:
df=pd.read_csv('all.csv')
df.head()

,0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,...,0.593,0.594,0.595,0.596,0.597,0.598,0.599,0.600,0.601,0.602
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [106]:
df.shape

(2999, 785)

In [107]:
df[['0.602']].sample(5)

,0.602
2713,9
2426,8
2126,7
508,1
1237,4


In [108]:
df=np.array(df)
np.random.shuffle(df)
df

array([[0, 0, 0, ..., 0, 0, 9],
       [0, 0, 0, ..., 0, 0, 3],
       [0, 0, 0, ..., 0, 0, 4],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 1],
       [0, 0, 0, ..., 0, 0, 3]], shape=(2999, 785))

In [109]:

dftrain=df[0:2000].T
X=dftrain[:-1]
Y=dftrain[-1]
dftest=df[2000:2999].T
X_test=dftest[:-1]
Y_test=dftest[-1]


In [110]:
X[:,0].shape

(784,)

In [111]:
l = 2   # number of layers
n = 10  # neurons per layer

def initParams(n,l):
    W = []
    b = []
    for i in range(0, l):
        if 0 == i:
            # Scale down the initial weights!
            W.append(np.random.randn(n, 784) * 0.01)
            b.append(np.zeros((n, 1))) # biases are safer starting at 0
        elif i == l - 1:
            # final output layer needs to match the 10 MNIST classes
            W.append(np.random.randn(10, n) * 0.01)
            b.append(np.zeros((10, 1)))
        else:
            W.append(np.random.randn(n, n) * 0.01)
            b.append(np.zeros((n, 1)))
    return W, b

def ReLU(Z):
    return np.maximum(0,Z)

def Softmax(Z):
    expZ = np.exp(Z - np.max(Z, axis=0, keepdims=True))
    return expZ / np.sum(expZ, axis=0, keepdims=True)

def for_prop(X, W, b, l):
    Z = []
    A = [X]
    for i in range(l):
        Z_curr = W[i].dot(A[i]) + b[i]
        Z.append(Z_curr)
        if i == l - 1:
            A.append(Softmax(Z_curr))
        else:
            A.append(ReLU(Z_curr))
    return Z, A

def one_hot(Y):
    one_hot_Y=np.zeros((Y.size, Y.max()+1))
    one_hot_Y[np.arange(Y.size), Y]=1
    one_hot_Y=one_hot_Y.T
    return one_hot_Y

def bac_prop(W,l,Z,A,Y):
    m=Y.size
    one_hot_Y=one_hot(Y)
    dZ={}
    dW={}
    db={}
    dZ[l]=A[l]-one_hot_Y
    dW[l]=(1/m)*np.dot(dZ[l], A[l-1].T)
    db[l]=(1/m)*np.sum(dZ[l], axis=1, keepdims=True)
    for i in reversed(range(1, l)):
        dA=np.dot(W[i].T, dZ[i+1])
        dZ[i]=dA*(Z[i-1]>0)# Element-wise derivative for ReLU

        dW[i]=(1/m)*np.dot(dZ[i], A[i-1].T)
        db[i]=(1/m)*np.sum(dZ[i], axis=1, keepdims=True)
    return dW, db

def optimize(W, b, dW, db, alpha):
    #dW1=dW[1], dW2=dw[2]...
    #W1=W[0], W2=W[1]...
    for i in range(0,l):
        W[i]=W[i]-alpha*dW[i+1]
        b[i]=b[i]-alpha*db[i+1]
    return W, b
   
def get_predictions(Al):
    return np.argmax(Al, 0)

def get_accuracy(predictions, Y):
    #print(predictions, Y)
    return np.sum(predictions==Y) / Y.size

def gradient_descent(X, Y, iterations, alpha, n, l):
    W,b =initParams(n,l)
    for i in range(iterations):
        Z,A=for_prop(X, W, b, l)
        dW, db= bac_prop(W,l,Z,A,Y)
        W, b=optimize(W, b, dW, db, alpha)
        if (0==i%100):
            print("Iteration: ", i)
            print("Accuracy: ", get_accuracy(get_predictions(A[l]), Y))
    return W, b

In [117]:
W, b = gradient_descent(X,Y,1001 ,0.001, n, l)

Iteration:  0
Accuracy:  0.1075
Iteration:  100
Accuracy:  0.847
Iteration:  200
Accuracy:  0.899
Iteration:  300
Accuracy:  0.9265
Iteration:  400
Accuracy:  0.9465
Iteration:  500
Accuracy:  0.9615
Iteration:  600
Accuracy:  0.972
Iteration:  700
Accuracy:  0.981
Iteration:  800
Accuracy:  0.9855
Iteration:  900
Accuracy:  0.9935
Iteration:  1000
Accuracy:  0.995


In [113]:
def predict(X, W, b, l):
    _, A=for_prop(X, W, b, l)
    predictions=get_predictions(A[l])
    return predictions

In [118]:
def test_prediction(X, Y, W, b, l):
    predictions=predict(X, W, b, l)
    accuracy=get_accuracy(predictions, Y)
    print(f"Final Test Accuracy: {accuracy * 100:.2f}%")
    return predictions


In [115]:
print("Train Features Shape (X):", X.shape)           # (784, 2000)
print("Train Labels Shape (Y):", Y.shape)             # (2000,)
print("Test Features Shape (X_test):", X_test.shape)   # (784, 999)

Train Features Shape (X): (784, 2000)
Train Labels Shape (Y): (2000,)
Test Features Shape (X_test): (784, 999)


In [119]:
test_preds = test_prediction(X_test, Y_test, W, b, l)

Final Test Accuracy: 87.49%
